# Email-Based Job Approval (Extended) - End-to-End Test

Extends `test_email_approval.ipynb` to cover:

1. The `sync` service in `syft_bg` (required for email approval to see new jobs from DS).
2. **Manual approval**: DS submits a job → DO replies `approve` → job runs → DS gets results.
3. **Email auto-approval**: DS submits a job containing `params.json` → DO replies `auto-approve` → the job is approved and an api is created from it. The DS then calls the api with other params; the call is approved without any email interaction.
4. **API-driven**: turn a pending job into an api via `syft_bg.create_api_from_job(...)` and call it from the DS side.

### Prerequisites
- GCP project with **Gmail API** and **Pub/Sub API** enabled
- OAuth credentials JSON (`app_credentials.json`)
- DS token (`token_ds.json`)

## Configuration

Fill in your email addresses and the Project ID below.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import syft as sy
from syft_rds import login_do, login_ds
import tempfile
import uuid
import json

In [ ]:
# Load .env file
for line in Path(".env").read_text().splitlines():
    if line.strip() and not line.startswith("#"):
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())

DO_EMAIL = os.environ["DO_EMAIL"]
DS_EMAIL = os.environ["DS_EMAIL"]
GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]

APP_CREDENTIALS = Path(os.environ["APP_CREDENTIALS"])
TOKEN_DS = Path(os.environ["TOKEN_DS"])

assert APP_CREDENTIALS.exists(), "App credentials missing"
assert TOKEN_DS.exists(), "DS credentials missing"

## Step 1: Create DO Drive token from app credentials

This converts the OAuth app credentials to a user-authorized Drive token.
It will open a browser window for you to authorize.

In [ ]:

do_token_path = str(APP_CREDENTIALS.parent / "token-do-2.json")
# do_token_path = sy.credentials_to_token(
#     credentials_path=str(APP_CREDENTIALS),
#     output_path=do_token_path,
#     do_scopes=True
# )
# print(f"DO token saved to: {do_token_path}")

## Step 2: Login as Data Owner and Data Scientist

In [ ]:
do_client = login_do(email=DO_EMAIL, token_path=str(do_token_path))

In [ ]:
ds_client = login_ds(email=DS_EMAIL, token_path=str(TOKEN_DS))

## Step 2a: Optional — wipe prior state

In [ ]:
# do_client.delete_syftbox()
# ds_client.delete_syftbox()

## Step 3: Set up peer relationship

DS requests access to DO, then DO approves.

In [ ]:
ds_client.add_peer(DO_EMAIL)

do_client.load_peers()
do_client.approve_peer_request(DS_EMAIL, peer_must_exist=False)

print("Peer relationship established")

## Step 4: Set up `syft-bg`

In addition to `notify` and `email_approve`, we now also start the `sync` service — it keeps the DO-side SyftBox up to date with Drive so that new jobs submitted by DS are visible to the email-approval service. We also start the `approve` service, which runs the calls of your apis.

In [ ]:
import syft_bg

In [ ]:
syft_bg.reset()

In [ ]:
syft_bg.init(
    do_email=DO_EMAIL,
    token_path=do_token_path,
    settings={
        "email_approve": {"gcp_project_id": GCP_PROJECT_ID},
    },
)

In [ ]:
# syft_bg.config

In [ ]:
syft_bg.ensure_running(
    services=["sync", "notify", "email_approve", "approve"],
    restart=True,
)

In [ ]:
syft_bg.status

## Step 5: Create some datasets

In [ ]:
do_client.create_dataset(
    name="testdata",
    mock_path="data/mock.txt",
    private_path="data/private.txt",
    users=[DS_EMAIL],
)

In [ ]:
!uv run create_test_pdfs.py

In [ ]:
do_client.create_dataset(
    name="pdfdata",
    mock_path="data/manifest.csv",
    private_path="data/PDFs",
    summary="A dataset with some PDFs",
    readme_path="data/readme.md",
    tags=["test", "pdf"],
    users=[DS_EMAIL],
    upload_private=False,
    sync=True
)

## Part A — Manual approval via email reply (`approve`)

DS submits a job. The `notify` service emails the DO. The DO replies `approve`. The `email_approve` service runs and shares the results back. The job uses: 
1. A dataset hosted by the data owner
2. A simple library dependence (Pandas)
3. Output written to a file

In [ ]:
MANUAL_JOB_NAME = f"email_test_{uuid.uuid4().hex[:8]}"
MANUAL_DIR = Path(tempfile.mkdtemp(prefix="email_approve_manual_"))
manual_job = MANUAL_DIR / "main.py"
manual_job.write_text('''import os, json
from pathlib import Path
import syft as sy
from syft_rds import login_do, login_ds
import pandas as pd

dataset_files = sy.resolve_dataset_files_path("pdfdata")
print(f"dataset director: {dataset_files}")

result = {"message": "Hello from email-approved job!", 
          "num_PDFs": len(dataset_files)}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(manual_job),
    job_name=MANUAL_JOB_NAME,
    force_submission=True,
    dependencies=["pandas"]
)
print(f"Job '{MANUAL_JOB_NAME}' submitted to {DO_EMAIL}")

### Reply `approve` to the notification email

1. Open the **DO's Gmail inbox**.
2. Find the notification email for the job above.
3. Reply with: `approve`.

To reject instead, reply `deny <reason>`. Re-run the cell below until the status flips to `done`.

In [ ]:
print(f"Check {DO_EMAIL}'s inbox and reply 'approve' to the job notification.")
job = next((j for j in do_client.jobs if j.name == MANUAL_JOB_NAME), None)
print(f"job {job.status}" if job else "job not visible yet")

### DS fetches the result

In [ ]:
ds_job = next(j for j in ds_client.jobs if j.name == MANUAL_JOB_NAME)
assert ds_job.status == "done", f"expected 'done', got {ds_job.status}"
print(f"Job {ds_job.name}: {ds_job.status}")
print(ds_job.output_paths[0].read_text())

## Part B — Email auto-approval via `auto-approve` reply

The DS submits a job whose `code_path` is a folder with both `main.py` and `params.json`. The DO replies `auto-approve` to the notification email. This approves the current job **and** creates an api from it, so the DS can call it later with other params (`ds_client.api[...]`). Resubmitting the same whole job would wait for the DO.

Reference: `tests/unit/syft_bg/test_email_auto_approve_flow.py::test_email_auto_approve_creates_api_that_runs_later_calls`.

In [ ]:

def create_parameterized_job(main_file: str, params: dict) -> Path:
    project_dir = Path(tempfile.mkdtemp(prefix="email_auto_approve_"))
    (project_dir / main_file).write_text('''import os, json

with open("params.json") as f:
    params = json.load(f)

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump({"params": params, "status": "done"}, f)
'''
    )
    (project_dir / "params.json").write_text(json.dumps(params))
    return project_dir

In [ ]:
FIRST_JOB_NAME = f"auto_approve_{uuid.uuid4().hex[:8]}"
first_job = create_parameterized_job(main_file="main.py", params={"run": 1})

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(first_job),
    job_name=FIRST_JOB_NAME,
    entrypoint="main.py",
    force_submission=True,
)
print(f"Submitted {FIRST_JOB_NAME} (with params.json)")

### Reply `auto-approve` to the notification email

Find the new notification for `FIRST_JOB` in the DO inbox and reply with: `auto-approve`.

Re-run the cell below until `status` is `done`.

In [ ]:
job = next((j for j in do_client.job_client.jobs if j.name == FIRST_JOB_NAME), None)
print(f"job {job.status}" if job else "job not visible yet")

### Verify the api was created

In [ ]:
from syft_bg.approve.api_store import ApiStore

api_def = ApiStore(do_client.syftbox_folder, do_client.email).get(FIRST_JOB_NAME)

content_names = {e.relative_path for e in api_def.file_contents}
assert content_names == {"code/main.py", "run.sh"}, content_names
assert "code/params.json" in api_def.file_paths, api_def.file_paths
print(f"Api '{FIRST_JOB_NAME}' created: {api_def}")

### DS fetches results for the first job

In [ ]:
ds_first = next(j for j in ds_client.jobs if j.name == FIRST_JOB_NAME)
assert ds_first.status == "done", ds_first.status
result = json.loads(ds_first.output_paths[0].read_text())
print(f"DS received first-job result: {result}")

### Call the api with other params

The DS calls the api by the first job's name. The call sends only `params.json` values, no code. The running `approve` service should run it on its next tick.

In [ ]:
ds_client.sync()
call = ds_client.api[FIRST_JOB_NAME](run=2, block=False)
SECOND_JOB_NAME = call.name
print(f"Called api {FIRST_JOB_NAME!r}, job {SECOND_JOB_NAME}")

In [ ]:
job = next((j for j in do_client.job_client.jobs if j.name == SECOND_JOB_NAME), None)
print(f"job {job.status}" if job else "job not visible yet")

In [ ]:
ds_second = next(j for j in ds_client.jobs if j.name == SECOND_JOB_NAME)
assert ds_second.status == "done", ds_second.status
result_2 = json.loads(ds_second.output_paths[0].read_text())
print(f"DS received second-job result (api call): {result_2}")

## Part C — Create an api manually

Instead of replying `auto-approve` to an email, the DO can create an api directly with `syft_bg.create_api_from_job(...)`. This takes a pending job as a template: content-matches `run.sh` and the Python file, name-matches `config.yaml` and `code/params.json`, and defaults `peers` to the submitter. The job itself is not approved by this; the DO approves it, and the DS then calls the api.

In [ ]:
API_JOB_NAME = f"api_auto_{uuid.uuid4().hex[:8]}"
api_job = create_parameterized_job(main_file="code.py", params={"run": "api-run"})

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(api_job),
    job_name=API_JOB_NAME,
    entrypoint="code.py",
    force_submission=True,
)
print(f"Submitted {API_JOB_NAME} (pending, awaiting DO review)")

In [ ]:
pending = [j for j in do_client.jobs if j.name == API_JOB_NAME]
assert pending, f"DO does not yet see {API_JOB_NAME} — wait for the sync service to tick and re-run"
assert pending[0].status == "pending", pending[0].status

### Create an api from the job above and approve the job

In [ ]:
result = syft_bg.create_api_from_job(pending[0])
assert result.success, result.error
print(result)
pending[0].approve()

### Verify the api was created and the job ran

In [ ]:
api_def = ApiStore(do_client.syftbox_folder, do_client.email).get(API_JOB_NAME)
assert {e.relative_path for e in api_def.file_contents} == {"code/code.py", "run.sh"}
print(f"Api '{API_JOB_NAME}' created from the pending job: {api_def}")

In [ ]:
job = next((j for j in do_client.job_client.jobs if j.name == API_JOB_NAME), None)
print(f"job {job.status}" if job else "job not visible yet")

In [ ]:
ds_api = next(j for j in ds_client.jobs if j.name == API_JOB_NAME)
assert ds_api.status == "done", ds_api.status
api_result = json.loads(ds_api.output_paths[0].read_text())
print(f"DS received result of the approved job: {api_result}")

# The DS calls the api with other params
ds_client.sync()
api_call = ds_client.api[API_JOB_NAME](run="api-run-2")
print(f"DS received api call result: {json.loads(api_call.output_paths[0].read_text())}")

## Debugging: service logs

In [ ]:
for svc in ("sync", "notify", "email_approve"): # Also "approve"
    print(f"\n=== {svc} ===")
    syft_bg.logs(svc, n=20)

## Test Failing Email Approve Job

In [ ]:
FAIL_JOB_NAME = f"email_test_{uuid.uuid4().hex[:8]}"
FAIL_JOB_DIR = Path(tempfile.mkdtemp(prefix="email_approve_manual_"))
fail_job = FAIL_JOB_DIR / "main.py"
fail_job.write_text('''
import unknown_package

result = {"message": "Hello from email-approved job!"}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(fail_job),
    job_name=FAIL_JOB_NAME,
    force_submission=True,
)
print(f"Job '{FAIL_JOB_NAME}' submitted to {DO_EMAIL}")

In [ ]:
do_client.jobs

### Reply `approve` to the notification email

1. Open the **DO's Gmail inbox**.
2. Find the notification email for the job above.
3. Reply with: `approve`.

To reject instead, reply `deny <reason>`. Re-run the cell below until the status flips to `done`.

In [ ]:
print(f"Check {DO_EMAIL}'s inbox and reply 'approve' to the job notification.")
job = next((j for j in do_client.jobs if j.name == FAIL_JOB_NAME), None)
print(f"job {job.status}" if job else "job not visible yet")

# Submit 100 Jobs

### Step 1: Create Job Template

In [ ]:

JOB_CODE = '''import os, json

result = {"message": "hello from test job"}
os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f)
print(result)
'''

_job_dir = Path(tempfile.mkdtemp(prefix="syft_test_"))
TEST_JOB = _job_dir / "main.py"
TEST_JOB.write_text(JOB_CODE)
print(f"Job script: {TEST_JOB}")


### Step 2: Submit Jobs in a Loop

In [ ]:
N_JOBS = 40
JOB_PREFIX = "syft_test_job"

In [ ]:
for i in range(N_JOBS):
    ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(TEST_JOB),
    job_name=f"{JOB_PREFIX}_manual_{i:03d}",
    force_submission=True,
)

### Step 3: Verify if we have received all 100 emails in DO inbox

In [ ]:
do_client.jobs

## Cleanup

In [ ]:
# syft_bg.stop()

In [ ]:
!rm -rf data/PDFs

In [ ]:
!rm data/manifest.csv data/readme.md